In [2]:
import numpy as np 
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub

/kaggle/input/datasets/sashankgarg23/crop-dataset/standardized_crops.csv
/kaggle/input/datasets/sashankgarg23/crop-dataset/merged_crop_dataset.csv


In [3]:
df=pd.read_csv('/kaggle/input/datasets/sashankgarg23/crop-dataset/merged_crop_dataset.csv')

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder , OneHotEncoder
from sklearn.model_selection import train_test_split as tts

In [5]:
X=df.drop('label' , axis=1)
Y=df['label']

In [6]:
x_train , x_test , y_train , y_test = tts(X , Y , test_size=0.2)

In [7]:
le=LabelEncoder()
y_train_scaled = le.fit_transform(y_train)
y_test_scaled = le.fit_transform(y_test)

In [8]:
import xgboost

In [8]:
model = xgboost.XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,

    objective="multi:softprob",
    eval_metric="mlogloss")

In [9]:
model.fit(x_train , y_train_scaled)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=400, n_jobs=None,
              num_parallel_tree=None, ...)

In [10]:
y_pred = model.predict(x_test)

In [11]:
from sklearn.metrics import accuracy_score

In [12]:
accuracy_score(y_test_scaled , y_pred)

0.996969696969697

In [14]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
import xgboost

# 1. Enable CUDA on XGBoost
model = xgboost.XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    tree_method="hist",  # Enables GPU histogram training
    device="cuda",        # Directs training to T4 GPU
    random_state=42
)

param_grid = {
    "n_estimators": [200, 400, 600],
    "max_depth": [3, 6, 9],
    "learning_rate": [0.05, 0.1, 0.2],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# 2. Set n_jobs=1 to prevent GPU memory conflicts
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=1,            
    verbose=2
)

grid_search.fit(x_train, y_train_scaled)

Fitting 5 folds for each of 108 candidates, totalling 540 fits


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [18:24:32] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=3, n_estimators=200, subsample=0.8; total time=   5.9s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=3, n_estimators=200, subsample=0.8; total time=   5.5s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=3, n_estimators=200, subsample=0.8; total time=   5.5s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=3, n_estimators=200, subsample=0.8; total time=   5.4s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=3, n_estimators=200, subsample=0.8; total time=   5.5s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=3, n_estimators=200, subsample=1.0; total time=   5.4s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=3, n_estimators=200, subsample=1.0; total time=   5.4s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=3, n_estimators=200, subsample=1.0; total time=   5.4s
[CV] END colsample_bytree=0.8, learning_rate=0.05, max_depth=3, n_estima

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device='cuda',
                                     early_stopping_rounds=None,
                                     enable_categorical=False,
                                     eval_metric='mlogloss', feature_types=None,
                                     feature_weights=None, gamma=None,
                                     gr...
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=None,
                                     n_jobs=None, num_parallel_tree=None, ...),
             n_jobs=1,
             param_grid={'colsample_bytree': [0.8, 1.0],
                         'learning_rate': [0.05, 0.1, 0.2],
                         'max_depth': [3, 6, 9],
                         'n_estimators': [200, 400, 600],
                         'subsample': [0.8, 1.0]},
             scoring='accuracy', verbose=2)

In [ ]:
{'colsample_bytree': 0.8,
 'learning_rate': 0.05,
 'max_depth': 3,
 'n_estimators': 200,
 'subsample': 1.0}

In [16]:
grid_search.best_params_

{'colsample_bytree': 0.8,
 'learning_rate': 0.05,
 'max_depth': 3,
 'n_estimators': 200,
 'subsample': 1.0}

In [17]:
print("Best Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)

Best Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200, 'subsample': 1.0}
Best CV Score: 0.9967783283067643


In [1]:
{'colsample_bytree': 0.8,
 'learning_rate': 0.05,
 'max_depth': 3,
 'n_estimators': 200,
 'subsample': 1.0}

{'colsample_bytree': 0.8,
 'learning_rate': 0.05,
 'max_depth': 3,
 'n_estimators': 200,
 'subsample': 1.0}

In [9]:
model = xgboost.XGBClassifier(
    n_estimators=400,
    max_depth=3,
    learning_rate=0.05,
    subsample=1.0,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss"
)

In [25]:
Y_encoded = le.fit_transform(Y)

In [26]:
model.fit(X,Y_encoded)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=400, n_jobs=None,
              num_parallel_tree=None, ...)

In [27]:
y_pred=model.predict(X)

In [28]:
from sklearn.metrics import accuracy_score

In [31]:
accuracy_score(Y_encoded,y_pred) 

1.0

In [32]:
import joblib

In [ ]:
import joblib
# Save model and label encoder
# Save model and label encoder
joblib.dump(model, "../../models/xgb_crop_model.joblib")
joblib.dump(le, "../../models/label_encoder.joblib")


In [3]:
import joblib

# Load model and label encoder
loaded_model = joblib.load("../../models/xgb_crop_model.joblib")
loaded_label_encoder = joblib.load("../../models/label_encoder.joblib")

c:\Users\KAILASH\anaconda3\envs\virtual\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [4]:
import numpy as np
import pandas as pd
sample_inputs = pd.DataFrame([
    {"N": 90, "P": 42, "K": 43, "temperature": 23.6, "humidity": 82.3, "ph": 6.5, "rainfall": 236.0},
    {"N": 20, "P": 134, "K": 199, "temperature": 22.3, "humidity": 92.3, "ph": 5.9, "rainfall": 110.0},
    {"N": 40, "P": 68, "K": 79, "temperature": 18.2, "humidity": 16.8, "ph": 7.3, "rainfall": 78.0},
    {"N": 78, "P": 46, "K": 42, "temperature": 25.0, "humidity": 80.0, "ph": 6.7, "rainfall": 175.0},
    {"N": 100, "P": 18, "K": 50, "temperature": 25.5, "humidity": 85.0, "ph": 6.4, "rainfall": 52.0}
])
pred = loaded_model.predict(sample_inputs)
predicted_crops = loaded_label_encoder.inverse_transform(pred)
sample_inputs["Predicted_Crop"] = predicted_crops
print(sample_inputs[["N", "P", "K", "temperature","ph", "rainfall", "Predicted_Crop"]])

     N    P    K  temperature   ph  rainfall Predicted_Crop
0   90   42   43         23.6  6.5     236.0           rice
1   20  134  199         22.3  5.9     110.0          apple
2   40   68   79         18.2  7.3      78.0       chickpea
3   78   46   42         25.0  6.7     175.0           jute
4  100   18   50         25.5  6.4      52.0     watermelon


In [ ]:
y_pred=loaded_model.predict(X)
accuracy_score(Y_encoded,y_pred) 